In [ ]:
import sys
!{sys.executable} -m pip install -q "gradio>=4.44.1" easyocr tensorflow scikit-learn pandas matplotlib

In [ ]:
import gradio as gr
print(gr.__version__)

5.50.0


In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import easyocr
import gradio as gr

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Optional: OpenCV is typically installed in Colab
import cv2

In [ ]:
# Download compressed OpenFoodFacts data (English subset)
# If this fails or is too large, you can download manually and upload to Colab instead.
!wget -O openfoodfacts_en.csv.gz "https://static.openfoodfacts.org/data/en.openfoodfacts.org.products.csv.gz"

import gzip

# We'll read directly from the gzip via pandas (no need to fully uncompress to disk)
cols_to_use = [
    "product_name",
    "energy-kcal_100g",
    "fat_100g",
    "saturated-fat_100g",
    "carbohydrates_100g",
    "sugars_100g",
    "proteins_100g",
    "salt_100g",
    "sodium_100g"
]

# Read a subset of rows for speed
n_rows = 100000  # change if you want more or less
df = pd.read_csv(
    "openfoodfacts_en.csv.gz",
    compression="gzip",
    sep="\t",
    usecols=cols_to_use,
    nrows=n_rows,
    low_memory=False
)

df.head()

--2025-12-07 03:57:36--  https://static.openfoodfacts.org/data/en.openfoodfacts.org.products.csv.gz
Resolving static.openfoodfacts.org (static.openfoodfacts.org)... 213.36.253.214
Connecting to static.openfoodfacts.org (static.openfoodfacts.org)|213.36.253.214|:443... connected.
HTTP request sent, awaiting response... 302 Moved Temporarily
Location: https://openfoodfacts-ds.s3.eu-west-3.amazonaws.com/en.openfoodfacts.org.products.csv.gz [following]
--2025-12-07 03:57:37--  https://openfoodfacts-ds.s3.eu-west-3.amazonaws.com/en.openfoodfacts.org.products.csv.gz
Resolving openfoodfacts-ds.s3.eu-west-3.amazonaws.com (openfoodfacts-ds.s3.eu-west-3.amazonaws.com)... 3.5.204.214, 3.5.205.205
Connecting to openfoodfacts-ds.s3.eu-west-3.amazonaws.com (openfoodfacts-ds.s3.eu-west-3.amazonaws.com)|3.5.204.214|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1189216488 (1.1G) [application/gzip]
Saving to: ‘openfoodfacts_en.csv.gz’

openfoodfacts_en.cs 100%[===============

,product_name,energy-kcal_100g,fat_100g,saturated-fat_100g,carbohydrates_100g,sugars_100g,proteins_100g,salt_100g,sodium_100g
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
cols = [
    "product_name", "energy-kcal_100g", "fat_100g",
    "saturated-fat_100g", "carbohydrates_100g",
    "sugars_100g", "proteins_100g", "salt_100g", "sodium_100g"
]

clean_df = df[cols].dropna().reset_index(drop=True)
clean_df.head()

,product_name,energy-kcal_100g,fat_100g,saturated-fat_100g,carbohydrates_100g,sugars_100g,proteins_100g,salt_100g,sodium_100g
0,granola Bio le Chocolaté,1.0,1.00,1.000,1.0,1.00,1.00,1.0000,0.4000
1,xytitol pastilles,70.0,0.50,0.060,2.0,0.24,18.00,0.2750,0.1100
2,Powdered peanut butter,45.0,13.00,6.700,15.0,3.60,11.00,0.0625,0.0250
3,Madeleines ChocoLait,460.0,24.00,6.000,54.0,31.00,6.40,0.4800,0.1920
4,Collagen For Her,123.0,1.76,0.882,24.7,0.00,1.76,0.0882,0.0353


In [ ]:
# Drop rows where almost everything is missing
feature_cols = [
    "energy-kcal_100g",
    "fat_100g",
    "saturated-fat_100g",
    "carbohydrates_100g",
    "sugars_100g",
    "proteins_100g",
    "salt_100g",
    "sodium_100g"
]

df_features = df[feature_cols].copy()

# Convert to numeric and handle errors
for col in feature_cols:
    df_features[col] = pd.to_numeric(df_features[col], errors="coerce")

# Drop rows with too many NaNs (here: require at least 5 non-null features)
df_features = df_features.dropna(thresh=5)

# For remaining NaNs, fill with column median (simple imputation)
for col in feature_cols:
    median_val = df_features[col].median()
    df_features[col] = df_features[col].fillna(median_val)

# Health label generation (rule-based -> 0/1/2)
def rule_based_health_label(row):
    score = 0

    energy = row["energy-kcal_100g"]
    sugar = row["sugars_100g"]
    fat = row["fat_100g"]
    sat_fat = row["saturated-fat_100g"]
    salt = row["salt_100g"]
    proteins = row["proteins_100g"]

    # Energy (per 100g)
    if energy > 500:
        score += 2
    elif energy > 300:
        score += 1

    # Sugar (per 100g)
    if sugar > 25:
        score += 2
    elif sugar > 10:
        score += 1

    # Fat
    if fat > 20:
        score += 2
    elif fat > 10:
        score += 1

    # Saturated fat
    if sat_fat > 10:
        score += 2
    elif sat_fat > 5:
        score += 1

    # Salt (g per 100g)
    if salt > 1.5:
        score += 2
    elif salt > 0.9:
        score += 1

    # Reward for protein (more protein = slightly better)
    if proteins >= 15:
        score -= 1

    # Map score to label: 0=Healthy, 1=Moderate, 2=Unhealthy
    if score <= 1:
        return 0  # Healthy
    elif score <= 4:
        return 1  # Moderate
    else:
        return 2  # Unhealthy

health_labels = df_features.apply(rule_based_health_label, axis=1)
df_features["health_label"] = health_labels

df_features["health_label"].value_counts()

,count
health_label,
0,33957
1,26221
2,18020


In [ ]:
X = df_features[feature_cols].values
y = df_features["health_label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

num_features = X_train_scaled.shape[1]
num_classes = 3  # Healthy / Moderate / Unhealthy

model = keras.Sequential([
    layers.Input(shape=(num_features,)),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    X_train_scaled,
    y_train,
    epochs=10,
    batch_size=256,
    validation_split=0.2,
    verbose=1
)

Epoch 1/10
196/196 ━━━━━━━━━━━━━━━━━━━━ 10s 39ms/step - accuracy: 0.6376 - loss: 0.8316 - val_accuracy: 0.8448 - val_loss: 0.3959
Epoch 2/10
196/196 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.8413 - loss: 0.4107 - val_accuracy: 0.8783 - val_loss: 0.3136
Epoch 3/10
196/196 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.8680 - loss: 0.3464 - val_accuracy: 0.8971 - val_loss: 0.2733
Epoch 4/10
196/196 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.8858 - loss: 0.2969 - val_accuracy: 0.9084 - val_loss: 0.2347
Epoch 5/10
196/196 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8993 - loss: 0.2632 - val_accuracy: 0.9184 - val_loss: 0.2130
Epoch 6/10
196/196 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9053 - loss: 0.2482 - val_accuracy: 0.9215 - val_loss: 0.1992
Epoch 7/10
196/196 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9114 - loss: 0.2296 - val_accuracy: 0.9263 - val_loss: 0.1896
Epoch 8/10
196/196 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9134 - loss: 0.2164 - val_accuracy:

In [ ]:
test_loss, test_acc = model.evaluate(X_test_scaled, y_test, verbose=0)
print("Test accuracy:", test_acc)

y_pred_probs = model.predict(X_test_scaled)
y_pred = np.argmax(y_pred_probs, axis=1)

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Test accuracy: 0.9304347634315491
489/489 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.97      0.96      6792
           1       0.90      0.89      0.90      5244
           2       0.93      0.90      0.92      3604

    accuracy                           0.93     15640
   macro avg       0.93      0.92      0.93     15640
weighted avg       0.93      0.93      0.93     15640

Confusion Matrix:
[[6608  184    0]
 [ 313 4684  247]
 [   0  344 3260]]


In [ ]:
label_map = {0: "Healthy", 1: "Moderate", 2: "Unhealthy"}

# Optionally save model & scaler for later use
model.save("nutrition_health_ann.h5")

import joblib
joblib.dump(scaler, "nutrition_scaler.joblib")

print("Saved model and scaler.")

Saved model and scaler.


In [ ]:
def parse_float_safe(val, default=None):
    try:
        return float(val)
    except Exception:
        return default

def extract_nutrition_from_text(text):
    """
    Very simple regex-based parser.
    Tries to extract calories, fat, saturated fat, carbs, sugar, protein, salt/sodium.
    This is a demo; real labels can be more complex.
    """
    lower = text.lower()

    def find_number(pattern, unit_multiplier=1.0):
        match = re.search(pattern, lower)
        if match:
            num = parse_float_safe(match.group(1))
            if num is not None:
                return num * unit_multiplier
        return None

    # energy (calories per serving or per 100g) – we treat as kcal_100g approx
    calories = find_number(r'calories?\s*([\d\.]+)')

    # total fat in grams
    fat = find_number(r'total\s*fat\s*([\d\.]+)\s*g') or find_number(r'fat\s*([\d\.]+)\s*g')

    # saturated fat
    sat_fat = find_number(r'saturated\s*fat\s*([\d\.]+)\s*g') or find_number(r'sat\.?\s*fat\s*([\d\.]+)\s*g')

    # total carbs
    carbs = find_number(r'(total\s*carbohydrate|carbohydrates?)\s*([\d\.]+)\s*g')
    if carbs is None:
        match = re.search(r'carbs?\s*([\d\.]+)\s*g', lower)
        if match:
            carbs = parse_float_safe(match.group(1))

    # sugars
    sugar = find_number(r'sugars?\s*([\d\.]+)\s*g')

    # protein
    protein = find_number(r'protein\s*([\d\.]+)\s*g')

    # salt (g) or sodium (mg/g)
    salt = find_number(r'salt\s*([\d\.]+)\s*g')
    sodium_mg = find_number(r'sodium\s*([\d\.]+)\s*mg', unit_multiplier=1.0)
    if sodium_mg is None:
        sodium_g = find_number(r'sodium\s*([\d\.]+)\s*g', unit_multiplier=1000.0)
        sodium_mg = sodium_g

    nutrition = {
        "energy-kcal_100g": calories,
        "fat_100g": fat,
        "saturated-fat_100g": sat_fat,
        "carbohydrates_100g": carbs,
        "sugars_100g": sugar,
        "proteins_100g": protein,
        "salt_100g": salt,
        "sodium_100g": sodium_mg
    }

    return nutrition

In [ ]:
def predict_health_ann(nutrition_dict):
    """
    Use the trained ANN + scaler to classify Healthy/Moderate/Unhealthy.
    For missing values, we fall back to global medians from df_features.
    """
    # Build feature vector in correct order
    x_vals = []
    for col in feature_cols:
        val = nutrition_dict.get(col)
        if val is None:
            # Use median from training data as fallback
            val = df_features[col].median()
        x_vals.append(val)

    x_array = np.array(x_vals).reshape(1, -1)
    x_scaled = scaler.transform(x_array)
    probs = model.predict(x_scaled, verbose=0)
    label_idx = int(np.argmax(probs, axis=1)[0])
    label_name = label_map[label_idx]
    return label_name, probs[0]

In [ ]:
def analyze_label_image(image):
    """
    1) Run OCR on the image
    2) Parse nutrition values
    3) Predict health class using ANN
    """
    if image is None:
        return None, None, None, None

    # Gradio gives image as numpy array (H, W, C)
    result = reader.readtext(image, detail=0)
    full_text = "\n".join(result)

    nutrition = extract_nutrition_from_text(full_text)
    label, probs = predict_health_ann(nutrition)

    return nutrition, label, probs, full_text

In [ ]:
def format_nutrition_text(nutrition):
    lines = []
    for k, v in nutrition.items():
        if v is not None:
            lines.append(f"{k}: {v}")
        else:
            lines.append(f"{k}: not detected")
    return "\n".join(lines)

def handle_image_upload(image, history, state):
    """
    Called when user uploads and clicks 'Analyze Label'.
    Returns updated chat history and state (nutrition info).
    """
    if image is None:
        bot_msg = "I couldn't read the image. Please upload a clear nutrition label."
        history.append({"role": "assistant", "content": bot_msg})
        return history, state

    nutrition, label, probs, raw_text = analyze_label_image(image)

    if nutrition is None:
        bot_msg = "I couldn't read the image. Please upload a clear nutrition label."
        history.append({"role": "assistant", "content": bot_msg})
        return history, state

    state = {
        "nutrition": nutrition,
        "label": label,
        "probs": probs,
        "raw_text": raw_text
    }

    nutri_str = format_nutrition_text(nutrition)
    prob_str = ", ".join(
        [f"{label_map[i]}: {probs[i]:.2f}" for i in range(len(probs))]
    )

    bot_msg = (
        "I've analyzed your nutrition label.\n\n"
        f"**Health classification (ANN):** **{label}**\n\n"
        f"**Class probabilities:** {prob_str}\n\n"
        f"**Extracted nutrients (approx):**\n{nutri_str}\n\n"
        "You can now ask me questions like:\n"
        "- Is this okay for daily use?\n"
        "- Is this suitable for someone with diabetes or high blood pressure?\n"
        "- Is there too much sugar or sodium?\n"
        "- How does this look overall in terms of health?\n"
    )
    history.append({"role": "user", "content": "[Image uploaded]"})
    history.append({"role": "assistant", "content": bot_msg})
    return history, state


def handle_user_message(user_message, history, state):
    """
    Chatbot reply based on last analyzed nutrition info + ANN label.
    """
    if not user_message:
        return history, state

    history.append({"role": "user", "content": user_message})

    if state is None or "nutrition" not in state:
        bot_msg = (
            "First upload a nutrition label image so I can analyze it. "
            "Then I can answer questions about how healthy it is."
        )
        history.append({"role": "assistant", "content": bot_msg})
        return history, state

    nutrition = state["nutrition"]
    label = state["label"]
    probs = state["probs"]

    cals = nutrition.get("energy-kcal_100g")
    sugar = nutrition.get("sugars_100g")
    sodium_mg = nutrition.get("sodium_100g")
    fat = nutrition.get("fat_100g")
    protein = nutrition.get("proteins_100g")
    salt_g = nutrition.get("salt_100g")

    msg_lower = user_message.lower()
    extra_lines = []

    # Diabetes / sugar
    if "diabet" in msg_lower or "sugar" in msg_lower or "sweet" in msg_lower:
        if sugar is not None:
            if sugar > 25:
                extra_lines.append(
                    f"For someone with diabetes, this appears **high in sugar** (~{sugar:.1f} g per 100g). "
                    "Likely not suitable for frequent consumption."
                )
            elif sugar > 10:
                extra_lines.append(
                    f"Sugar is **moderate** (~{sugar:.1f} g per 100g). Occasional intake might be fine, "
                    "but it's not ideal for strict blood sugar control."
                )
            else:
                extra_lines.append(
                    f"Sugar level is relatively **low** (~{sugar:.1f} g per 100g), which is better for diabetes management."
                )
        else:
            extra_lines.append("I couldn't detect sugar clearly from the label.")

    # Blood pressure / sodium / salt
    if "pressure" in msg_lower or "bp" in msg_lower or "sodium" in msg_lower or "salt" in msg_lower:
        if sodium_mg is not None:
            if sodium_mg > 1000:
                extra_lines.append(
                    f"Sodium is **very high** (~{sodium_mg:.0f} mg per 100g). Not recommended for high blood pressure."
                )
            elif sodium_mg > 500:
                extra_lines.append(
                    f"Sodium is **moderate to high** (~{sodium_mg:.0f} mg per 100g). It should be limited for high BP."
                )
            else:
                extra_lines.append(
                    f"Sodium is **relatively lower** (~{sodium_mg:.0f} mg per 100g) compared to many processed foods."
                )
        elif salt_g is not None:
            if salt_g > 1.5:
                extra_lines.append(
                    f"Salt is **high** (~{salt_g:.2f} g per 100g), which corresponds to high sodium. "
                    "Not ideal for high blood pressure."
                )
            else:
                extra_lines.append(
                    f"Salt level (~{salt_g:.2f} g per 100g) is moderate."
                )
        else:
            extra_lines.append("I couldn't clearly detect sodium or salt from the label.")

    # Daily consumption
    if "daily" in msg_lower or "every day" in msg_lower or "day" in msg_lower:
        if label == "Healthy":
            extra_lines.append(
                "The ANN classifies this as **Healthy**, so in reasonable portions it can be okay for daily use."
            )
        elif label == "Moderate":
            extra_lines.append(
                "This is **Moderate**. It's fine occasionally, but for daily use you may prefer options with lower sugar, fat, or salt."
            )
        else:
            extra_lines.append(
                "This is classified as **Unhealthy**. I would not recommend it for daily consumption."
            )

    # Calories / weight / general health
    if "calorie" in msg_lower or "weight" in msg_lower or "diet" in msg_lower:
        if cals is not None:
            if cals > 400:
                extra_lines.append(
                    f"Calories (~{cals:.0f} kcal per 100g) are relatively high, so portion control is important if you're watching your weight."
                )
            elif cals > 250:
                extra_lines.append(
                    f"Calories (~{cals:.0f} kcal per 100g) are moderate; not extremely high but still significant."
                )
            else:
                extra_lines.append(
                    f"Calories (~{cals:.0f} kcal per 100g) are relatively low compared to many processed snacks."
                )
        else:
            extra_lines.append("I couldn't accurately detect the calorie information.")

    # If nothing specific matched, send a general explanation
    if not extra_lines:
        prob_str = ", ".join(
            [f"{label_map[i]}: {probs[i]:.2f}" for i in range(len(probs))]
        )
        extra_lines.append(
            f"In general, this product is classified as **{label}** by the ANN. "
            f"Class probabilities: {prob_str}"
        )

    bot_msg = "\n\n".join(extra_lines)
    history.append({"role": "assistant", "content": bot_msg})
    return history, state

In [ ]:
# Initialize EasyOCR reader (run once)
reader = easyocr.Reader(['en'])

In [ ]:
custom_css = """
/* Make layout similar to ChatGPT */
.gradio-container {
    max-width: 850px !important;
    margin: auto !important;
    padding-top: 30px;
    background: #f7f7f8;
    font-family: 'Inter', sans-serif;
}

/* Chat messages like ChatGPT */
#chatbot {
    height: 520px !important;
    background: white !important;
    border-radius: 12px !important;
    padding: 20px !important;
    overflow-y: auto !important;
    border: 1px solid #e6e6e6 !important;
}

.message.user {
    background: #dbeafe !important;
    color: #0f172a !important;
    padding: 12px 16px !important;
    border-radius: 12px !important;
    max-width: 80%;
    margin-left: auto !important;
    margin-bottom: 12px;
}

.message.bot {
    background: #f1f5f9 !important;
    color: #000 !important;
    padding: 12px 16px !important;
    border-radius: 12px !important;
    max-width: 80%;
    margin-right: auto !important;
    margin-bottom: 12px;
}

/* Input area like ChatGPT */
.input-row {
    position: sticky;
    bottom: 0;
    background: #f7f7f8;
    padding: 10px 0;
}

.textbox textarea {
    border-radius: 14px !important;
    padding: 12px !important;
    background: white !important;
    border: 1px solid #d4d4d4 !important;
}

/* Buttons */
.analyze-btn {
    background: #3b82f6 !important;
    color: white !important;
    padding: 10px 18px !important;
    border-radius: 10px !important;
}

.send-btn {
    background: #2563eb !important;
    color: white !important;
    padding: 10px 18px !important;
    border-radius: 10px !important;
}
"""

with gr.Blocks(css=custom_css) as demo: # Moved css=custom_css back here

    gr.Markdown("<h2 style='text-align:center;'>🥗 AI Nutrition Label ChatGPT</h2>")

    state = gr.State(None)

    chatbot = gr.Chatbot(
        label="Chat",
        elem_id="chatbot",
        type="messages",
        allow_tags=False # Explicitly set to False to prevent future warning
        # Removed bubble_full_width=False as it is deprecated
    )

    # Image + Analyze row (simple & clean)
    with gr.Row():
        image_input = gr.Image(
            label="Upload Nutrition Label",
            type="numpy",
            height=180
        )
        analyze_btn = gr.Button("🔍 Analyze Label", elem_classes="analyze-btn")

    # Input row like ChatGPT
    with gr.Row(elem_classes="input-row"):
        msg = gr.Textbox(
            placeholder="Send a message...",
            lines=2,
            scale=4,
            elem_classes="textbox"
        )
        send_btn = gr.Button("Send", elem_classes="send-btn", scale=1)

    analyze_btn.click(
        fn=handle_image_upload,
        inputs=[image_input, chatbot, state],
        outputs=[chatbot, state]
    )

    send_btn.click(
        fn=handle_user_message,
        inputs=[msg, chatbot, state],
        outputs=[chatbot, state]
    )

/tmp/ipython-input-391665502.py:72: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css) as demo: # Moved css=custom_css back here
/tmp/ipython-input-391665502.py:78: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(


In [ ]:
demo.launch(
    share=True,
    debug=False, # Changed to False to mitigate RuntimeError
    show_error=True,
    max_threads=1
    # Removed css=custom_css from here
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://1466b53065ce2178cd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/h11_impl.py", line 403, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1133, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 113, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py",